# Wakefield

IMPACT-Z applies a read-in short-range wakefield through the zero-length `RfcavityStructureWakefield` element (type `-41`). The element behaves as a switch: it enables the wake for every element that follows it, until a second `-41` element switches it back off.

This example

1. builds a wakefield object with [openPMD-beamphysics](https://github.com/ChristopherMayes/openPMD-beamphysics),
2. writes it out in the IMPACT-Z table format,
3. tracks a Gaussian bunch through a one meter drift with the wake enabled, and
4. compares the resulting energy loss with the wake potential computed directly from the same wakefield object.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from beamphysics import ParticleGroup
from beamphysics.interfaces.impact import create_impact_z_wakefield_rfdata
from beamphysics.units import c_light, mec2
from beamphysics.wakefields import TabularWakefield

import impact.z as IZ

## The wakefield model

The short-range longitudinal wake of a periodic disk-loaded accelerating structure is well described by the analytic form of Bane, SLAC-PUB-11829, Eq. (12):

$$W_\parallel(s) = \frac{Z_0 c}{\pi a^2} \exp\left(-\sqrt{s / s_1}\right),
\qquad
s_1 = 0.41 \, \frac{a^{1.8} g^{1.6}}{L^{2.4}}$$

with iris radius $a$, gap $g$ and period $L$. The values below are those of the SLAC S-band linac structure, and are the same ones used in the Impact-T wakefield example.

In [ ]:
Z0 = 376.730313668  # Ohm

a = 0.0116  # iris radius (m)
g = 0.0292  # gap (m)
L = 0.035  # period (m)

s1 = 0.41 * a**1.8 * g**1.6 / L**2.4


def bane_wake(s):
    """Longitudinal wake in V/C/m at a distance s >= 0 behind the source particle."""
    return Z0 * c_light / (np.pi * a**2) * np.exp(-np.sqrt(np.abs(s) / s1))


s1

openPMD-beamphysics represents a wakefield as a function of $z \le 0$, where $z$ is the position of the trailing particle relative to the source. Any of the classes in `beamphysics.wakefields` may be used here; `TabularWakefield` simply interpolates supplied data, which is the usual situation when the wake comes from an electromagnetic solver or a measurement.

In [ ]:
# Largest trailing distance to tabulate. This must comfortably cover the bunch.
ZMAX = 0.01  # m

s_table = np.linspace(0, ZMAX, 5000)

wake = TabularWakefield(-s_table[::-1], bane_wake(s_table[::-1]))
wake

In [ ]:
wake.plot(zmax=1e-3)

## Building the IMPACT-Z wake table

The `-41` element reads its wake from `rfdata{file_id}.in`, a four-column table on a uniform grid:

| Column | Quantity | Units |
| --- | --- | --- |
| 1 | distance $s$ behind the source particle | m |
| 2 | longitudinal wake $W_\parallel(s)$ | V/C/m |
| 3 | horizontal dipole wake $W_x(s)$ | V/C/m$^2$ |
| 4 | vertical dipole wake $W_y(s)$ | V/C/m$^2$ |

A positive longitudinal wake means the trailing particle loses energy, which is the same sign convention that openPMD-beamphysics uses. The only difference is the direction of the abscissa. `create_impact_z_wakefield_rfdata` performs the conversion, and enforces the 5000 point limit that is compiled into IMPACT-Z. It returns the table as an array, which is what `ImpactZInput.file_data` expects; `write_impact_z_wakefield` writes the same table to a file for use outside lume-impact.


In [ ]:
rfdata = create_impact_z_wakefield_rfdata(wake, zmax=ZMAX)
rfdata.shape

In [ ]:
rfdata[:5]

## Initial particles

A 1 nC, 1 GeV Gaussian bunch with $\sigma_z = 100\ \mu\mathrm{m}$.

In [ ]:
N_PARTICLE = 100_000
CHARGE = 1e-9  # C
ENERGY = 1e9  # eV
SIGMA_Z = 100e-6  # m
SIGMA_X = 100e-6  # m

rng = np.random.default_rng(42)
z = rng.normal(0, SIGMA_Z, N_PARTICLE)

P0 = ParticleGroup(
    data=dict(
        x=rng.normal(0, SIGMA_X, N_PARTICLE),
        y=rng.normal(0, SIGMA_X, N_PARTICLE),
        z=np.zeros(N_PARTICLE),
        px=np.zeros(N_PARTICLE),
        py=np.zeros(N_PARTICLE),
        pz=np.full(N_PARTICLE, np.sqrt(ENERGY**2 - mec2**2)),
        t=-z / c_light,
        weight=np.full(N_PARTICLE, CHARGE / N_PARTICLE),
        status=np.ones(N_PARTICLE),
        species="electron",
    )
)
P0

## Lattice and run

Three details matter when setting up a wakefield run:

- **The beam current must be nonzero.** IMPACT-Z skips its entire collective effects section when the current is zero, and the wake is computed there. Setting `bunch_charge` (equivalently `average_current`) is therefore required.
- **Space charge can be disabled separately** with a `ToggleSpaceCharge` element, which leaves the wakefield active. That isolates the wake for this comparison.
- **The bunch must have a finite transverse size.** The wake is deposited on the space-charge mesh, and a bunch with zero transverse extent produces a degenerate computational domain.

The longitudinal wake is resolved on the `nz` mesh, so `nz` controls the accuracy of the result.

In [ ]:
L_DRIFT = 1.0  # m
FILE_ID = 41

input = IZ.ImpactZInput(
    ncpu_y=1,
    ncpu_z=1,
    seed=0,
    n_particle=N_PARTICLE,
    integrator_type="linear_map",
    nx=32,
    ny=32,
    nz=256,
    boundary_type="trans_open_longi_open",
    radius_x=1.0,
    radius_y=1.0,
    distribution="read",
    reference_kinetic_energy=ENERGY - mec2,
    reference_particle_mass=mec2,
    reference_particle_charge=-1.0,
    reference_frequency=1e9,
    file_data={str(FILE_ID): rfdata},
    lattice=[
        IZ.ToggleSpaceCharge(enable=False),
        IZ.WriteFull(file_id=1000, name="initial_particles"),
        IZ.RfcavityStructureWakefield(
            file_id=FILE_ID, enable_wakefield=1.0, name="wake_on"
        ),
        IZ.Drift(length=L_DRIFT, steps=10, map_steps=1, radius=1.0, name="d1"),
        IZ.RfcavityStructureWakefield(
            file_id=FILE_ID, enable_wakefield=-1.0, name="wake_off"
        ),
        IZ.WriteFull(file_id=2000, name="final_particles"),
    ],
)

# Nonzero current is required for the wakefield to be computed.
input.bunch_charge = CHARGE
input.average_current

In [ ]:
print(input.lattice[2].to_line())

In [ ]:
I = IZ.ImpactZ(input, initial_particles=P0)
I.run()

In [ ]:
P1 = I.output.particles["final_particles"]
P1

In [ ]:
P1.plot("delta_t", "delta_pz")

## Comparison with the wake potential

`WakefieldBase.convolve_density` computes the wake potential $V(z)$ in V/m from a longitudinal charge density in C/m, using the same causal convolution and half self-kick that IMPACT-Z uses. The momentum change over a length $\ell$ is $\Delta p_z c = -\ell V(z)$.

In [ ]:
z0 = -c_light * P0["t"]

counts, edges = np.histogram(z0, bins=400, weights=P0["weight"])
z_bin = 0.5 * (edges[1:] + edges[:-1])
dz = z_bin[1] - z_bin[0]
density = counts / dz  # C/m

wake_potential = wake.convolve_density(density, dz, plot=True)  # V/m

In [ ]:
# IMPACT-Z does not preserve the particle ordering.
order = np.argsort(P1["id"])
assert np.all(P1["id"][order] == P0["id"])

delta_pz = P1["pz"][order] - P0["pz"]
delta_pz_expected = -L_DRIFT * np.interp(z0, z_bin, wake_potential)

print(f"IMPACT-Z mean energy loss: {-delta_pz.mean() * 1e-3:.3f} keV")
print(f"Expected mean energy loss: {-delta_pz_expected.mean() * 1e-3:.3f} keV")

In [ ]:
fig, ax = plt.subplots()
ax.scatter(
    z0[::20] * 1e6,
    delta_pz[::20] * 1e-3,
    marker="x",
    s=8,
    alpha=0.3,
    label="IMPACT-Z tracking",
)
isort = np.argsort(z0)
ax.plot(
    z0[isort] * 1e6,
    delta_pz_expected[isort] * 1e-3,
    color="red",
    label="openPMD-beamphysics wake potential",
)
ax.set_xlabel(r"$z$ (µm)  (head to the right)")
ax.set_ylabel(r"$\Delta p_z c$ (keV)")
ax.set_title(f"Wake-induced energy change over {L_DRIFT} m")
ax.legend()

The residual is dominated by the linear deposition of the charge onto the `nz` mesh. IMPACT-Z versions before 2.7.8 take the value of the next table row instead of interpolating, which raises the residual by about a factor of four here.

In [ ]:
core = np.abs(z0) < 3 * SIGMA_Z
rel = np.std(delta_pz[core] - delta_pz_expected[core]) / np.std(delta_pz_expected[core])
print(f"relative rms difference in the core: {rel:.2e}")

## Scaling and transverse wakes

`RfcavityStructureWakefield.scale` multiplies the tabulated wake, which is convenient when the same table describes a structure that appears with different fill factors.

`enable_wakefield` selects which components act:

| Value | Effect |
| --- | --- |
| `-1.0` | wake off |
| `1.0` | longitudinal wake only |
| `10.0` | longitudinal and transverse dipole wakes |

The transverse dipole wakes are supplied in columns three and four of the table, and may be passed to `create_impact_z_wakefield_rfdata` through `wake_x` and `wake_y`.


## Other wakefield models

Because only the wake function is needed, any openPMD-beamphysics wakefield can be exported the same way. A resistive-wall wake, for example:

In [ ]:
from beamphysics.wakefields import ResistiveWallWakefield

rw_wake = ResistiveWallWakefield.from_material("copper-slac-pub-10707", radius=2.5e-3)
rw_rfdata = create_impact_z_wakefield_rfdata(rw_wake, zmax=1e-3)

rw_wake.plot(zmax=1e-3)

## Recovering the wakefield from an input

The conversion runs in both directions. `TabularWakefield.from_impact_z` accepts either a path to an `rfdata{N}.in` file or the table itself, so a wakefield read from an existing IMPACT-Z deck, or restored from a lume-impact archive, can be turned back into a model for plotting and analysis.


In [ ]:
recovered = TabularWakefield.from_impact_z(I.input.file_data[str(FILE_ID)])

np.allclose(recovered.wake(wake.z_data), wake.wake(wake.z_data))